# Comprehensive Pre‑processing Evaluation on the IoT Intrusion Detection Dataset

This notebook iterates over a set of classic data‑pre‑processing techniques... For every technique we:
1️⃣ Apply it to a **copy** of the original CSV (the source file is never modified).
2️⃣ Train a simple **DecisionTreeClassifier** on a stratified 70/30 train‑test split.
3️⃣ Record key performance metrics – Accuracy, Precision, Recall, F1‑score – using the original target column `label` (assumed binary).
4️⃣ Append the results to a summary table shown at the end of the notebook.

> **Note**: Some integration‑related techniques (e.g., schema matching) require multiple data sources and are therefore illustrated with a placeholder comment; they are still listed for completeness.

In [2]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Path to the original dataset (do not modify)
DATA_PATH = Path(r'D:/work/dmProj/AI_Powered_IoT_Network_Intrusion_Detection_Dataset.csv')

# Load once – the original dataframe stays untouched
original_df = pd.read_csv(DATA_PATH)

# Identify the target column – adjust if your dataset uses a different name
TARGET_COL = 'intrusion_label'  # <-- change if needed

# Helper to evaluate a model on a given dataframe
def evaluate(df):
    # Fix: Only use numeric features for the classifier to avoid conversion errors
    X = df.drop(columns=[TARGET_COL]).select_dtypes(include=[np.number])
    y = df[TARGET_COL]
    
    # Handle edge case where no numeric features exist
    if X.empty:
        return {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0}
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
    clf = DecisionTreeClassifier(random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0)
    }

# Container for results
results = []


## 1️⃣ Data Cleaning – Raza

### Mean Imputation
*Definition*: Replace missing numeric values with the column mean.
*Pros*: Simple, fast, preserves dataset size.
*Cons*: Sensitive to outliers; can distort variance.
*When suitable*: Small amount of MCAR (Missing Completely at Random) numeric gaps, and you need a quick baseline.
*When NOT suitable*: Presence of strong outliers or when preserving original distribution is critical.

In [ ]:
df = original_df.copy()
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].mean()) # Use assignment
metrics = evaluate(df)
metrics['technique'] = 'Mean Imputation'
results.append(metrics)


### Median Imputation
*Definition*: Replace missing numeric values with the column median.
*Pros*: Robust to outliers, simple.
*Cons*: Still discards distribution shape; not suitable for categorical missingness.
*When suitable*: Numeric data with outliers or skewed distributions where mean would be misleading.

In [ ]:
df = original_df.copy()
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
metrics = evaluate(df)
metrics['technique'] = 'Median Imputation'
results.append(metrics)

### Mode Imputation
*Definition*: Replace missing values (numeric or categorical) with the most frequent value.
*Pros*: Works for categorical data, preserves mode.
*Cons*: Can bias towards majority class, may create unrealistic duplicates.
*When suitable*: Categorical columns with few categories, or numeric columns where the mode is meaningful.

In [ ]:
df = original_df.copy()
for col in df.columns:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
metrics = evaluate(df)
metrics['technique'] = 'Mode Imputation'
results.append(metrics)

### Deletion Method
*Definition*: Remove any record that contains at least one missing value.
*Pros*: Guarantees no missing data, no imputation bias.
*Cons*: Can dramatically reduce dataset size, especially with many features.
*When suitable*: Small amount of missingness or when preserving data integrity is paramount.

In [ ]:
df = original_df.dropna()
metrics = evaluate(df)
metrics['technique'] = 'Deletion'
results.append(metrics)

### Interquartile Range (IQR) Outlier Removal
*Definition*: Discard rows where any numeric value lies outside 
`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
*Pros*: Simple, works well for symmetric distributions.
*Cons*: May remove legitimate extreme values; only handles univariate outliers.
*When suitable*: When you suspect measurement errors causing extreme points.

In [9]:

df = original_df.copy()
num = df.select_dtypes(include='number')
Q1 = num.quantile(0.25)
Q3 = num.quantile(0.75)
IQR = Q3 - Q1
# Added bridge parenthesis after (Q3 + 1.5 * IQR)
mask = ~((num < (Q1 - 1.5 * IQR)) | (num > (Q3 + 1.5 * IQR))).any(axis=1) 
df = df[mask]
metrics = evaluate(df)
metrics['technique'] = 'IQR Outlier Removal'
results.append(metrics)
df.head(3)


,device_id,device_type,firmware_version,connection_type,packet_count,avg_packet_size,bandwidth_usage_mb,failed_connection_attempts,unusual_port_access,protocol_type,login_attempts,encryption_used,anomaly_score,geo_location,traffic_spike_ratio,intrusion_label
0,D1,Smart Plug,v1.1,5G,22904,941,3572.07,4,0,TCP,12,1,0.439,Europe,0.90,0
1,D2,Router,v3.0,Ethernet,36382,887,275.60,6,0,MQTT,13,1,0.278,Asia,1.86,0
6,D7,Sensor,v2.1,WiFi,49817,1226,4168.04,3,0,HTTP,22,1,0.365,North America,4.35,0


### Z‑Score Outlier Removal
*Definition*: Remove rows where any numeric feature has a |z| > 3.
*Pros*: Captures multivariate extreme points, standard statistical rule.
*Cons*: Assumes normality; may miss outliers in heavy‑tailed data.
*When suitable*: When features are roughly Gaussian or after standardisation.

In [27]:
df = original_df.copy()
num = df.select_dtypes(include='number')
z = (num - num.mean()) / num.std()
mask = (z.abs() <= 3).all(axis=1)
df = df[mask]
metrics = evaluate(df)
metrics['technique'] = 'Z‑Score Outlier Removal'
results.append(metrics)
original_df.head(5)
df.head()

,device_id,device_type,firmware_version,connection_type,packet_count,avg_packet_size,bandwidth_usage_mb,failed_connection_attempts,unusual_port_access,protocol_type,login_attempts,encryption_used,anomaly_score,geo_location,traffic_spike_ratio,intrusion_label
0,D1,Smart Plug,v1.1,5G,22904,941,3572.07,4,0,TCP,12,1,0.439,Europe,0.90,0
1,D2,Router,v3.0,Ethernet,36382,887,275.60,6,0,MQTT,13,1,0.278,Asia,1.86,0
2,D3,Sensor,v3.0,WiFi,10971,577,3803.74,11,0,TCP,7,0,0.618,North America,0.42,0
4,D5,Smart Plug,v2.1,Ethernet,23197,485,4167.80,11,1,HTTP,12,1,0.463,Asia,0.73,0
5,D6,Router,v3.0,Ethernet,275,580,1098.13,8,1,TCP,26,1,0.380,Asia,1.13,0


### Binning
*Definition*: Discretise a continuous variable into a fixed number of bins (here 5).
*Pros*: Reduces noise, can capture non‑linear relationships, works well with tree models.
*Cons*: Information loss, choice of bin count is heuristic.
*When suitable*: When the variable is highly skewed or you want categorical interactions.

In [25]:
df = original_df.copy()
if 'packet_count' in df.columns:
    df['packet_count_bin'] = pd.cut(df['packet_count'], bins=5, labels=False)
metrics = evaluate(df)
metrics['technique'] = 'Binning (packet_count)'
results.append(metrics)

df.head(5)

,device_id,device_type,firmware_version,connection_type,packet_count,avg_packet_size,bandwidth_usage_mb,failed_connection_attempts,unusual_port_access,protocol_type,login_attempts,encryption_used,anomaly_score,geo_location,traffic_spike_ratio,intrusion_label,packet_count_bin
0,D1,Smart Plug,v1.1,5G,22904,941,3572.07,4,0,TCP,12,1,0.439,Europe,0.90,0,2
1,D2,Router,v3.0,Ethernet,36382,887,275.60,6,0,MQTT,13,1,0.278,Asia,1.86,0,3
2,D3,Sensor,v3.0,WiFi,10971,577,3803.74,11,0,TCP,7,0,0.618,North America,0.42,0,1
3,D4,Smart Plug,v2.0,5G,13636,925,2886.37,19,0,MQTT,13,1,0.738,Europe,0.73,1,1
4,D5,Smart Plug,v2.1,Ethernet,23197,485,4167.80,11,1,HTTP,12,1,0.463,Asia,0.73,0,2


### Regression Smoothing (Iterative Imputer)
*Definition*: Use a regression model (here Bayesian Ridge) to predict missing values iteratively.
*Pros*: Captures multivariate relationships, generally better than simple stats.
*Cons*: Computationally heavier, may overfit if data is small.
*When suitable*: When missingness is moderate and you suspect dependencies among features.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
df = original_df.copy()
num = df.select_dtypes(include='number')
imp = IterativeImputer(random_state=42, max_iter=10)
df[num.columns] = imp.fit_transform(num)
metrics = evaluate(df)
metrics['technique'] = 'Regression Smoothing (Iterative Imputer)'
results.append(metrics)

### Duplicate Detection
*Definition*: Remove rows that are exact duplicates of another row.
*Pros*: Eliminates redundant data, reduces storage.
*Cons*: Might drop legitimate repeated observations if duplication is meaningful.
*When suitable*: When data collection pipelines can produce identical logs.

In [ ]:
df = original_df.drop_duplicates()
metrics = evaluate(df)
metrics['technique'] = 'Duplicate Removal'
results.append(metrics)

## 2️⃣ Data Integration – Eishal (place‑holder)
The following steps normally require two or more source tables.  In this single‑file example we only note the intent and skip execution.
*Schema Matching*, *Entity Resolution*, *Correlation Analysis*, *Data Conflict Resolution*, *Duplicate Elimination* are listed for completeness.

## 3️⃣ Data Transformation – Tayyaba

### Min‑Max Normalization
*Definition*: Scale each numeric feature to the **[0, 1]** range.
*Pros*: Preserves shape, works well for distance‑based models.
*Cons*: Sensitive to outliers.
*When suitable*: When features have different units and you need bounded inputs.

In [ ]:
df = original_df.copy()
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = (df[num_cols] - df[num_cols].min()) / (df[num_cols].max() - df[num_cols].min())
metrics = evaluate(df)
metrics['technique'] = 'Min‑Max Normalization'
results.append(metrics)

### Z‑Score Normalization
*Definition*: Center each numeric feature to mean 0 and std 1.
*Pros*: Handles different scales, works with many algorithms.
*Cons*: Assumes roughly Gaussian distribution; outliers affect scaling.
*When suitable*: When you plan to use linear models or algorithms sensitive to variance.

In [ ]:
df = original_df.copy()
# The fixed logic:
num_cols = [c for c in df.select_dtypes(include='number').columns if c != TARGET_COL]
df[num_cols] = (df[num_cols] - df[num_cols].mean()) / df[num_cols].std()
metrics = evaluate(df)
metrics['technique'] = 'Z‑Score Normalization'
results.append(metrics)

### Decimal Scaling
*Definition*: Move the decimal point of each value based on the maximum absolute value (e.g., divide by 10^j).
*Pros*: Simple, retains sign.
*Cons*: Choice of j can be arbitrary; less common.
*When suitable*: When you need a quick rough scaling without computing min/max.

In [ ]:
df = original_df.copy()
num_cols = df.select_dtypes(include='number').columns
j = np.ceil(np.log10(df[num_cols].abs().max()))
df[num_cols] = df[num_cols] / (10 ** j)
metrics = evaluate(df)
metrics['technique'] = 'Decimal Scaling'
results.append(metrics)

### Log Transformation
*Definition*: Apply `log(x + 1)` (or similar) to reduce right‑skew.
*Pros*: Compresses large values, can make distributions more Gaussian.
*Cons*: Cannot handle zero/negative values without shifting; may lose interpretability.
*When suitable*: Highly skewed features such as counts or sizes.

In [ ]:
df = original_df.copy()
# The fixed logic:
num_cols = [c for c in df.select_dtypes(include='number').columns if c != TARGET_COL]
df[num_cols] = np.log1p(df[num_cols].clip(lower=0))
metrics = evaluate(df)
metrics['technique'] = 'Log Transformation'
results.append(metrics)

### One‑Hot Encoding
*Definition*: Convert each categorical level into a binary column.
*Pros*: No ordinal assumptions, works with tree‑based and linear models.
*Cons*: Can explode dimensionality for high‑cardinality features.
*When suitable*: Low‑cardinality categorical columns.

In [ ]:
df = original_df.copy()
cat_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
metrics = evaluate(df)
metrics['technique'] = 'One‑Hot Encoding'
results.append(metrics)

### Label Encoding
*Definition*: Map each categorical value to a unique integer.
*Pros*: Simple, low memory.
*Cons*: Implicit ordinal relationship may mislead some models.
*When suitable*: Tree‑based models that can handle integer categories, or when cardinality is very high.

In [ ]:
df = original_df.copy()
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes
metrics = evaluate(df)
metrics['technique'] = 'Label Encoding'
results.append(metrics)

### Aggregation
*Definition*: Group rows by a key (e.g., time window or device) and compute summary statistics.
*Pros*: Reduces data size, can reveal higher‑level patterns.
*Cons*: May discard granular information; requires a meaningful grouping key.
*When suitable*: When the raw granularity is too fine‑grained for the model.

In [ ]:
df = original_df.copy()
# Example: aggregate by 'src_ip' if present
if 'src_ip' in df.columns:
    agg = df.groupby('src_ip').mean().reset_index()
    df = agg
else:
    df = original_df  # fallback – no aggregation possible
metrics = evaluate(df)
metrics['technique'] = 'Aggregation (src_ip mean)'
results.append(metrics)

## 4️⃣ Data Reduction – Armaghan

### Principal Component Analysis (PCA)
*Definition*: Linear projection onto orthogonal components that maximise variance.
*Pros*: Reduces dimensionality, removes multicollinearity.
*Cons*: Components are linear combos – interpretability drops; assumes linear relationships.
*When suitable*: When you have many correlated numeric features and want a compact representation.

In [ ]:
from sklearn.decomposition import PCA
df = original_df.copy()
num_cols = df.select_dtypes(include='number').columns
pca = PCA(n_components=0.95, random_state=42)  # keep 95% variance
principal = pca.fit_transform(df[num_cols])
pca_df = pd.DataFrame(principal, columns=[f'pc{i+1}' for i in range(principal.shape[1])])
# Preserve the target column
pca_df[TARGET_COL] = df[TARGET_COL].values
metrics = evaluate(pca_df)
metrics['technique'] = 'PCA (95% variance)'
results.append(metrics)

### Linear Discriminant Analysis (LDA)
*Definition*: Projects data onto a lower‑dimensional space that maximises class separability.
*Pros*: Supervised dimensionality reduction, often improves classification.
*Cons*: Works only for classification tasks with labeled data; assumes normality & equal covariances.
*When suitable*: When you have a moderate number of classes and want a discriminative feature set.

In [20]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

df = original_df.copy()

# Remove identifier column
df = df.drop(columns=["device_id"])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)

# Apply LDA
lda = LinearDiscriminantAnalysis(n_components=None)
X_lda = lda.fit_transform(X, y)

# Create dataframe for evaluation
lda_df = pd.DataFrame(X_lda, columns=[f'lda{i+1}' for i in range(X_lda.shape[1])])
lda_df[TARGET_COL] = y.values

# Evaluate
metrics = evaluate(lda_df)
metrics['technique'] = 'LDA'
results.append(metrics)

### Filter Methods (Univariate Feature Selection)
*Definition*: Rank features by statistical tests (e.g., chi‑square) and keep the top‑k.
*Pros*: Fast, model‑agnostic.
*Cons*: Ignores feature interactions.
*When suitable*: When you need a quick reduction before heavier modeling.

In [24]:
from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd

df = original_df.copy()

# Remove identifier column
df = df.drop(columns=["device_id"])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)

# Convert boolean columns to integers
X = X.astype(int)

# Ensure non-negative values for chi2
X_nonneg = X - X.min() + 1

# Select top 10 features
selector = SelectKBest(score_func=chi2, k=10)
X_new = selector.fit_transform(X_nonneg, y)

cols = [f'feat{i+1}' for i in range(X_new.shape[1])]
filter_df = pd.DataFrame(X_new, columns=cols)
filter_df[TARGET_COL] = y.values

metrics = evaluate(filter_df)
metrics['technique'] = 'Filter Method (SelectKBest chi2 k=10)'
results.append(metrics)

### Wrapper Methods (Recursive Feature Elimination)
*Definition*: Iteratively train a model (here DecisionTree) and prune the least important features.
*Pros*: Takes feature interactions into account.
*Cons*: Computationally expensive, may overfit on small data.
*When suitable*: When you can afford extra training time for a more tuned feature set.

In [26]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

df = original_df.copy()

# Remove identifier column
df = df.drop(columns=["device_id"])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Convert categorical columns to numeric
X = pd.get_dummies(X, drop_first=True)

# Convert boolean columns to integers
X = X.astype(int)

clf = DecisionTreeClassifier(random_state=42)

rfe = RFE(estimator=clf, n_features_to_select=10, step=1)
X_rfe = rfe.fit_transform(X, y)

rfe_df = pd.DataFrame(X_rfe, columns=[f'rfe{i+1}' for i in range(X_rfe.shape[1])])
rfe_df[TARGET_COL] = y.values

metrics = evaluate(rfe_df)
metrics['technique'] = 'Wrapper (RFE DecisionTree k=10)'
results.append(metrics)

### Embedded Methods (Tree‑based Feature Importance)
*Definition*: Train a model that provides inherent feature importance (e.g., RandomForest) and keep the top‑k.
*Pros*: Joint learning + selection; relatively fast.
*Cons*: Dependent on the chosen estimator; may miss features useful for other models.
*When suitable*: When you plan to use tree‑based models downstream.

In [28]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

df = original_df.copy()

# Remove identifier column
df = df.drop(columns=["device_id"])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Convert categorical features to numeric
X = pd.get_dummies(X, drop_first=True)

# Convert boolean columns to integers
X = X.astype(int)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Get top 10 important features
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]

X_top = X.iloc[:, indices]

embed_df = X_top.copy()
embed_df[TARGET_COL] = y.values

metrics = evaluate(embed_df)
metrics['technique'] = 'Embedded (RF top-10)'
results.append(metrics)

### Simple Random Sampling
*Definition*: Randomly select a subset of rows (e.g., 30%).
*Pros*: Reduces size, fast to compute.
*Cons*: May not preserve class distribution; variance in results.
*When suitable*: When the dataset is huge and you need a quick prototype.

In [29]:
df = original_df.sample(frac=0.3, random_state=42)
metrics = evaluate(df)
metrics['technique'] = 'Simple Random Sampling (30%)'
results.append(metrics)

### Stratified Sampling
*Definition*: Sample rows while preserving the proportion of each class label.
*Pros*: Keeps class balance, better for classification performance estimation.
*Cons*: Slightly more complex; still reduces total data.
*When suitable*: When the dataset is imbalanced and you still want a smaller subset.

In [30]:
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.7, random_state=42)
for _, idx in sss.split(original_df, original_df[TARGET_COL]):
    df_strat = original_df.iloc[idx]
metrics = evaluate(df_strat)
metrics['technique'] = 'Stratified Sampling (30%)'
results.append(metrics)

## Summary of Results

In [31]:
summary_df = pd.DataFrame(results)
# Re‑order columns for readability
summary_df = summary_df[['technique','accuracy','precision','recall','f1']]
summary_df.reset_index(drop=True, inplace=True)
summary_df

,technique,accuracy,precision,recall,f1
0,Mean Imputation,0.818333,0.095238,0.103448,0.099174
1,Median Imputation,0.818333,0.095238,0.103448,0.099174
2,Mode Imputation,0.818333,0.095238,0.103448,0.099174
3,Deletion,0.818333,0.095238,0.103448,0.099174
4,IQR Outlier Removal,1.000000,0.000000,0.000000,0.000000
5,Z‑Score Outlier Removal,1.000000,0.000000,0.000000,0.000000
6,Binning (packet_count),0.815417,0.101887,0.116379,0.108652
7,Regression Smoothing (Iterative Imputer),0.818333,0.095238,0.103448,0.099174
8,Duplicate Removal,0.818333,0.095238,0.103448,0.099174
9,Min‑Max Normalization,0.818333,0.095238,0.103448,0.099174
